Task 2 – Interactive Choropleth Map

## Objective

To visualize global installs for the top 5 app categories using an interactive Choropleth Map.

## Dataset

Google Play Store Dataset

## Transformations Applied

* Cleaned Install counts and converted to numeric format
* Excluded categories beginning with A, C, G, and S
* Selected top 5 categories based on total installs
* Assigned representative countries to categories for visualization purposes

## KPI Measured

* Total Installs by Category
* Categories with installs exceeding 1 million

## Visualization

Interactive Plotly Choropleth Map with hover information and install-based color intensity.

## Note

The original dataset does not contain geographical information. Representative countries were assigned to categories solely for visualization purposes.

## Time Restriction

The visualization is available only between 6 PM IST and 8 PM IST.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_csv(
    r"C:\Users\HP\Downloads\archive\googleplaystore.csv"
)

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [3]:
df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace(',', '')
    .str.replace('+', '')
)

df['Installs'] = pd.to_numeric(
    df['Installs'],
    errors='coerce'
)

In [4]:
filtered_df = df[
    ~df['Category'].str.startswith(
        ('A','C','G','S'),
        na=False
    )
]

filtered_df.shape

(8161, 13)

In [5]:
top5 = (
    filtered_df
    .groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

top5

Category
PRODUCTIVITY          1.417609e+10
TOOLS                 1.145277e+10
FAMILY                1.025826e+10
PHOTOGRAPHY           1.008825e+10
NEWS_AND_MAGAZINES    7.496318e+09
Name: Installs, dtype: float64

In [6]:
filtered_df = filtered_df[
    filtered_df['Category'].isin(top5.index)
]

filtered_df['Category'].value_counts()

Category
FAMILY                1972
TOOLS                  843
PRODUCTIVITY           424
PHOTOGRAPHY            335
NEWS_AND_MAGAZINES     283
Name: count, dtype: int64

In [7]:
country_map = {
    top5.index[0]: 'United States',
    top5.index[1]: 'India',
    top5.index[2]: 'Germany',
    top5.index[3]: 'Canada',
    top5.index[4]: 'Australia'
}

country_map

{'PRODUCTIVITY': 'United States',
 'TOOLS': 'India',
 'FAMILY': 'Germany',
 'PHOTOGRAPHY': 'Canada',
 'NEWS_AND_MAGAZINES': 'Australia'}

In [8]:
filtered_df['Country'] = (
    filtered_df['Category']
    .map(country_map)
)

filtered_df[['Category','Country']].head()

,Category,Country
2014,FAMILY,Germany
2015,FAMILY,Germany
2016,FAMILY,Germany
2017,FAMILY,Germany
2018,FAMILY,Germany


In [9]:
map_df = (
    filtered_df
    .groupby(['Country','Category'])['Installs']
    .sum()
    .reset_index()
)

map_df

,Country,Category,Installs
0,Australia,NEWS_AND_MAGAZINES,7.496318e+09
1,Canada,PHOTOGRAPHY,1.008825e+10
2,Germany,FAMILY,1.025826e+10
3,India,TOOLS,1.145277e+10
4,United States,PRODUCTIVITY,1.417609e+10


In [10]:
map_df['Highlight'] = np.where(
    map_df['Installs'] > 1000000,
    'Above 1 Million',
    'Below 1 Million'
)

map_df

,Country,Category,Installs,Highlight
0,Australia,NEWS_AND_MAGAZINES,7.496318e+09,Above 1 Million
1,Canada,PHOTOGRAPHY,1.008825e+10,Above 1 Million
2,Germany,FAMILY,1.025826e+10,Above 1 Million
3,India,TOOLS,1.145277e+10,Above 1 Million
4,United States,PRODUCTIVITY,1.417609e+10,Above 1 Million


The Google Play Store dataset does not contain geographical information.
To create the required choropleth visualization, representative countries were assigned to the top 5 app categories for demonstration purposes.

In [12]:
from datetime import datetime
import pytz
import plotly.express as px

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 18 <= current_time.hour < 20:

    fig = px.choropleth(
        map_df,
        locations='Country',
        locationmode='country names',
        color='Installs',
        hover_name='Category',
        hover_data=['Highlight'],
        title='Global Installs by Top 5 App Categories'
    )

    fig.show()

else:
    print(
        "Choropleth map is available only between 6 PM IST and 8 PM IST."
    )

Choropleth map is available only between 6 PM IST and 8 PM IST.
